In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sksurv.metrics import concordance_index_censored
from lifelines import CoxPHFitter
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

base   = '/Users/parthshringarpure/Desktop/AI/Projects/luad_survival'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Device: {device}")
print("All imports successful ✅")

Device: cpu
All imports successful ✅


In [2]:
import urllib.request
import os

# Download TCGA sample type annotations from Xena
# This file tells us which cancer type each patient belongs to
annotations_url = "https://tcga-pancan-atlas-hub.s3.us-east-1.amazonaws.com/download/Survival_SupplementalTable_S1_20171025_xena_sp"
annotations_path = f'{base}/data/external/tcga_sample_annotations.tsv'

if not os.path.exists(annotations_path):
    print("Downloading TCGA sample annotations...")
    urllib.request.urlretrieve(annotations_url, annotations_path)
    print("Downloaded ✅")
else:
    print("File already exists ✅")

# Load and inspect
annotations = pd.read_csv(annotations_path, sep='\t')
print(f"\nAnnotations shape: {annotations.shape}")
print(f"Columns: {list(annotations.columns)}")
print(f"\nFirst 3 rows:")
print(annotations.head(3))

Downloaded ✅

Annotations shape: (12591, 34)
Columns: ['sample', '_PATIENT', 'cancer type abbreviation', 'age_at_initial_pathologic_diagnosis', 'gender', 'race', 'ajcc_pathologic_tumor_stage', 'clinical_stage', 'histological_type', 'histological_grade', 'initial_pathologic_dx_year', 'menopause_status', 'birth_days_to', 'vital_status', 'tumor_status', 'last_contact_days_to', 'death_days_to', 'cause_of_death', 'new_tumor_event_type', 'new_tumor_event_site', 'new_tumor_event_site_other', 'new_tumor_event_dx_days_to', 'treatment_outcome_first_course', 'margin_status', 'residual_tumor', 'OS', 'OS.time', 'DSS', 'DSS.time', 'DFI', 'DFI.time', 'PFI', 'PFI.time', 'Redaction']

First 3 rows:
            sample      _PATIENT cancer type abbreviation  \
0  TCGA-OR-A5J1-01  TCGA-OR-A5J1                      ACC   
1  TCGA-OR-A5J2-01  TCGA-OR-A5J2                      ACC   
2  TCGA-OR-A5J3-01  TCGA-OR-A5J3                      ACC   

   age_at_initial_pathologic_diagnosis  gender   race  \
0      

In [3]:
# Check available cancer types
print("All cancer types in TCGA:")
print(annotations['cancer type abbreviation'].value_counts())

All cancer types in TCGA:
cancer type abbreviation
BRCA    1236
KIRC     944
LUAD     641
LUSC     623
OV       604
HNSC     604
GBM      602
UCEC     583
THCA     580
PRAD     566
COAD     545
LGG      529
STAD     511
SKCM     479
LIHC     438
BLCA     436
KIRP     352
CESC     312
SARC     271
ESCA     204
LAML     200
PAAD     196
PCPG     187
READ     183
TGCT     139
THYM     126
ACC       92
KICH      91
MESO      87
UVM       80
UCS       57
DLBC      48
CHOL      45
Name: count, dtype: int64


In [4]:
# Filter to lung cancer patients only (LUAD + LUSC)
lung_annotations = annotations[
    annotations['cancer type abbreviation'].isin(['LUAD', 'LUSC'])
].copy()

print(f"LUAD patients: {(lung_annotations['cancer type abbreviation']=='LUAD').sum()}")
print(f"LUSC patients: {(lung_annotations['cancer type abbreviation']=='LUSC').sum()}")
print(f"Total lung:    {len(lung_annotations)}")

# Get sample IDs
lung_sample_ids = set(lung_annotations['sample'].tolist())
print(f"\nExample sample IDs: {list(lung_sample_ids)[:5]}")

# Remove our 478 LUAD training patients to avoid leakage
# Load our patient IDs
expr_luad = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', 
                         index_col=0)
our_patients = set(expr_luad.index)

# Convert our patient IDs to TCGA format (tcga-05-4249 → TCGA-05-4249-01)
# Pan-cancer uses uppercase with tumour suffix
our_patients_upper = set()
for p in our_patients:
    # tcga-05-4249 → TCGA-05-4249-01A (approximate)
    upper = p.upper() + '-01'
    our_patients_upper.add(upper)

print(f"\nOur LUAD patients: {len(our_patients)}")
print(f"Example our ID:    {list(our_patients_upper)[:3]}")

LUAD patients: 641
LUSC patients: 623
Total lung:    1264

Example sample IDs: ['TCGA-85-A510-01', 'TCGA-77-A5G3-01', 'TCGA-50-5049-01', 'TCGA-37-4130-01', 'TCGA-MP-A4TF-01']

Our LUAD patients: 478
Example our ID:    ['TCGA-55-8619-01', 'TCGA-69-7979-01', 'TCGA-50-5049-01']


In [5]:
# Load our 790 pretrained genes list
pretrain_genes = json.load(open(f'{base}/models/experiments/pretrain_genes.json'))
ensembl_to_symbol = json.load(open(f'{base}/models/ensembl_to_symbol.json'))

print("Reading pan-cancer TPM file for lung patients only...")
print("This will take 3-5 minutes...")

# Read full pan-cancer file
pancancer_raw = pd.read_csv(
    f'{base}/data/external/pancancer_tpm',
    sep='\t', index_col=0)

print(f"Full pan-cancer shape: {pancancer_raw.shape}")

# Filter to our 790 genes
pancancer_filtered = pancancer_raw.loc[
    pancancer_raw.index.isin(ensembl_to_symbol.keys())
]

# Rename to gene symbols
pancancer_filtered.index = [ensembl_to_symbol[g] for g in pancancer_filtered.index]

# Transpose: patients × genes
pancancer_filtered = pancancer_filtered.T
print(f"After filtering genes: {pancancer_filtered.shape}")

# Filter to lung patients only
lung_patients_in_pancancer = [p for p in pancancer_filtered.index 
                               if p in lung_sample_ids]
pancancer_lung = pancancer_filtered.loc[lung_patients_in_pancancer]
print(f"Lung patients found: {len(pancancer_lung)}")

# Remove our LUAD training patients
our_ids_in_pancancer = [p for p in pancancer_lung.index 
                         if p in our_patients_upper]
pancancer_lung = pancancer_lung.drop(index=our_ids_in_pancancer, errors='ignore')
print(f"After removing our LUAD patients: {len(pancancer_lung)}")

# Check cancer type breakdown
luad_in_pretrain = lung_annotations[
    (lung_annotations['sample'].isin(pancancer_lung.index)) & 
    (lung_annotations['cancer type abbreviation']=='LUAD')
].shape[0]
lusc_in_pretrain = lung_annotations[
    (lung_annotations['sample'].isin(pancancer_lung.index)) & 
    (lung_annotations['cancer type abbreviation']=='LUSC')
].shape[0]

print(f"\nBreakdown:")
print(f"  LUAD (not our patients): {luad_in_pretrain}")
print(f"  LUSC:                    {lusc_in_pretrain}")
print(f"  Total for pretraining:   {len(pancancer_lung)}")

Reading pan-cancer TPM file for lung patients only...
This will take 3-5 minutes...
Full pan-cancer shape: (60498, 10535)
After filtering genes: (10535, 790)
Lung patients found: 1122
After removing our LUAD patients: 647

Breakdown:
  LUAD (not our patients): 99
  LUSC:                    548
  Total for pretraining:   647


In [7]:
# Scale lung data
scaler_lung = StandardScaler()
lung_scaled = scaler_lung.fit_transform(pancancer_lung.values)

print(f"Lung pretraining data: {lung_scaled.shape}")
print(f"Mean: {lung_scaled.mean():.4f}")
print(f"Std:  {lung_scaled.std():.4f}")

# Dataset
class PancancerDataset(Dataset):
    def __init__(self, data):
        self.data = torch.FloatTensor(data)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

class ExpressionAutoencoder(nn.Module):
    def __init__(self, input_dim=790, latent_dim=32, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

# Train/val split
X_train, X_val = train_test_split(lung_scaled, test_size=0.1, random_state=42)

train_loader = DataLoader(PancancerDataset(X_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(PancancerDataset(X_val),   batch_size=64, shuffle=False)

print(f"\nTrain: {len(X_train)} patients")
print(f"Val:   {len(X_val)} patients")

# Pretrain
ae_lung = ExpressionAutoencoder(input_dim=790).to(device)
optimizer = torch.optim.Adam(ae_lung.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = nn.MSELoss()

best_val_loss    = np.inf
best_weights     = None
patience_counter = 0
patience         = 20
epochs           = 150

print(f"\nPretraining on {len(X_train)} lung cancer patients...")
print(f"{'Epoch':<8} {'Train Loss':<14} {'Val Loss':<14}")
print("-" * 36)

for epoch in range(1, epochs + 1):
    ae_lung.train()
    train_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, _ = ae_lung(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    ae_lung.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            recon, _ = ae_lung(batch)
            val_loss += criterion(recon, batch).item()
    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_weights     = {k: v.clone() for k, v in ae_lung.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f"{epoch:<8} {train_loss:<14.4f} {val_loss:<14.4f}")

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

ae_lung.load_state_dict(best_weights)
print(f"\nPretraining done! Best val loss: {best_val_loss:.4f}")

Lung pretraining data: (647, 790)
Mean: -0.0000
Std:  1.0000

Train: 582 patients
Val:   65 patients

Pretraining on 582 lung cancer patients...
Epoch    Train Loss     Val Loss      
------------------------------------
1        0.9557         0.7802        
10       0.6591         0.5149        
20       0.5890         0.4641        
30       0.5829         0.4392        
40       0.5478         0.4339        
50       0.5416         0.4290        
60       0.5301         0.4170        
70       0.5200         0.4146        
80       0.5211         0.4139        
90       0.5323         0.4156        
100      0.5131         0.4150        
110      0.5045         0.4113        
120      0.5209         0.4128        

Early stopping at epoch 122

Pretraining done! Best val loss: 0.4107


In [8]:
import os
os.makedirs(f'{base}/models/experiments/lung_pretrain', exist_ok=True)

# Save full autoencoder
torch.save(ae_lung.state_dict(), 
           f'{base}/models/experiments/lung_pretrain/autoencoder_lung.pt')

# Save encoder weights only
encoder_weights_lung = {k.replace('encoder.', ''): v 
                        for k, v in ae_lung.state_dict().items() 
                        if k.startswith('encoder.')}
torch.save(encoder_weights_lung, 
           f'{base}/models/experiments/lung_pretrain/encoder_weights_lung.pt')

# Save scaler
with open(f'{base}/models/experiments/lung_pretrain/scaler_lung.pkl', 'wb') as f:
    pickle.dump(scaler_lung, f)

print("Saved:")
print(f"  models/experiments/lung_pretrain/autoencoder_lung.pt")
print(f"  models/experiments/lung_pretrain/encoder_weights_lung.pt")
print(f"  models/experiments/lung_pretrain/scaler_lung.pkl")
print(f"\nPretrained on: {len(X_train)} lung cancer patients")
print(f"Best val loss: {best_val_loss:.4f}")

Saved:
  models/experiments/lung_pretrain/autoencoder_lung.pt
  models/experiments/lung_pretrain/encoder_weights_lung.pt
  models/experiments/lung_pretrain/scaler_lung.pkl

Pretrained on: 582 lung cancer patients
Best val loss: 0.4107


In [10]:
from sksurv.metrics import concordance_index_censored
from sklearn.model_selection import StratifiedKFold
from lifelines import CoxPHFitter

class SurvivalDataset(Dataset):
    def __init__(self, expr, dysreg, immune, clinical, times, events):
        self.expr     = torch.FloatTensor(expr)
        self.dysreg   = torch.FloatTensor(dysreg)
        self.immune   = torch.FloatTensor(immune)
        self.clinical = torch.FloatTensor(clinical)
        self.times    = torch.FloatTensor(times)
        self.events   = torch.FloatTensor(events)
    def __len__(self):
        return len(self.times)
    def __getitem__(self, idx):
        return (self.expr[idx], self.dysreg[idx],
                self.immune[idx], self.clinical[idx],
                self.times[idx], self.events[idx])

def cox_loss(risk_scores, times, events):
    order       = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order].squeeze()
    events      = events[order]
    log_cumsum  = torch.logcumsumexp(risk_scores, dim=0)
    return -torch.mean((risk_scores - log_cumsum)[events.bool()])

def train_model(model, train_loader, val_expr, val_dysreg,
                val_immune, val_clinical, val_times, val_events,
                epochs=300, patience=30, lr=0.001, noise=0.05):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    best_val_cindex  = 0
    best_weights     = None
    patience_counter = 0
    for epoch in range(epochs):
        model.train()
        for x_expr, x_dysreg, x_immune, x_clinical, times, events in train_loader:
            x_expr     = x_expr.to(device)
            x_dysreg   = x_dysreg.to(device)
            x_immune   = x_immune.to(device)
            x_clinical = x_clinical.to(device)
            times      = times.to(device)
            events     = events.to(device)
            x_expr   = x_expr   + torch.randn_like(x_expr)   * noise
            x_dysreg = x_dysreg + torch.randn_like(x_dysreg) * noise
            x_immune = x_immune + torch.randn_like(x_immune) * noise
            optimizer.zero_grad()
            risk, _ = model(x_expr, x_dysreg, x_immune, x_clinical)
            loss = cox_loss(risk, times, events)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_risk, _ = model(
                val_expr.to(device), val_dysreg.to(device),
                val_immune.to(device), val_clinical.to(device))
            val_risk = val_risk.squeeze().cpu().numpy()
        val_ci = concordance_index_censored(
            val_events.astype(bool), val_times, val_risk)[0]
        scheduler.step(-val_ci)
        if val_ci > best_val_cindex:
            best_val_cindex  = val_ci
            best_weights     = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= patience:
            break
    model.load_state_dict(best_weights)
    return model, best_val_cindex, epoch

class FusionModelPretrained(nn.Module):
    def __init__(self, expr_dim=790, dysreg_dim=20,
                 immune_dim=22, clinical_dim=5, dropout=0.5):
        super().__init__()
        self.encoder_expr = nn.Sequential(
            nn.Linear(expr_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32)
        )
        self.encoder_dysreg = nn.Sequential(
            nn.Linear(dysreg_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32)
        )
        self.encoder_immune = nn.Sequential(
            nn.Linear(immune_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )
        self.encoder_clinical = nn.Sequential(
            nn.Linear(clinical_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32)
        )
        self.attention = nn.Sequential(
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1)
        )
        self.output = nn.Linear(32, 1)

    def load_pretrained_encoder(self, weights_path):
        pretrained = torch.load(weights_path, map_location=device)
        self.encoder_expr.load_state_dict(pretrained)
        print("Pretrained encoder weights loaded ✅")

    def forward(self, x_expr, x_dysreg, x_immune, x_clinical):
        h_expr     = self.encoder_expr(x_expr)
        h_dysreg   = self.encoder_dysreg(x_dysreg)
        h_immune   = self.encoder_immune(x_immune)
        h_clinical = self.encoder_clinical(x_clinical)
        streams      = torch.stack([h_expr, h_dysreg, h_immune, h_clinical], dim=1)
        attn_weights = torch.softmax(self.attention(streams), dim=1)
        fused        = (attn_weights * streams).sum(dim=1)
        return self.output(fused), attn_weights.squeeze(-1)

print("All classes and functions defined ✅")

All classes and functions defined ✅


In [11]:
# Load LUAD data
expr     = pd.read_csv(f'{base}/data/processed/expression_matrix.csv', index_col=0)
dysreg   = pd.read_csv(f'{base}/data/processed/dysregulation_scores.csv', index_col=0)
immune   = pd.read_csv(f'{base}/data/processed/immune_features_cibersort.csv', index_col=0)
clinical = pd.read_csv(f'{base}/data/processed/clinical_survival.csv', index_col=0)

common   = expr.index.intersection(dysreg.index).intersection(
           immune.index).intersection(clinical.index)
expr     = expr.loc[common]
dysreg   = dysreg.loc[common]
immune   = immune.loc[common]
clinical = clinical.loc[common]

age           = clinical[['age']].copy()
gender        = (clinical['gender'] == 'male').astype(float).to_frame()
stage_dummies = pd.get_dummies(clinical['stage_group'], prefix='stage')
stage_dummies = stage_dummies.drop(columns=['stage_Stage I'], errors='ignore')
clinical_features = pd.concat([age, gender, stage_dummies], axis=1).astype(float).fillna(0)

y = np.array(
    [(bool(e), t) for e, t in zip(clinical['event'], clinical['survival_time'])],
    dtype=[('event', bool), ('time', float)]
)

# Filter expression to pretrained gene space
pretrain_genes = json.load(open(f'{base}/models/experiments/pretrain_genes.json'))
common_genes   = [g for g in pretrain_genes if g in expr.columns]
expr_790       = expr[common_genes]

print(f"Patients:         {len(common)}")
print(f"Expression genes: {expr_790.shape[1]}")
print(f"Events:           {y['event'].sum()} ({y['event'].mean()*100:.1f}%)")

# ── CV loop ─────────────────────────────────────────────────────────
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_cindex       = []
fold_attn_weights = []

print("\nRunning leakage-free 5-fold CV — Lung Pretrained Fusion Model")
print(f"{'Fold':<6} {'Best Epoch':<12} {'Test C-index':<12}")
print("-" * 32)

for fold, (train_idx, test_idx) in enumerate(kf.split(expr_790, y['event']), 1):

    expr_train,     expr_test     = expr_790.iloc[train_idx],          expr_790.iloc[test_idx]
    dysreg_train,   dysreg_test   = dysreg.iloc[train_idx],            dysreg.iloc[test_idx]
    immune_train,   immune_test   = immune.iloc[train_idx],            immune.iloc[test_idx]
    clinical_train, clinical_test = clinical_features.iloc[train_idx], clinical_features.iloc[test_idx]
    y_train,        y_test        = y[train_idx],                      y[test_idx]

    times_train  = y_train['time'].copy()
    events_train = y_train['event'].copy()
    times_test   = y_test['time'].copy()
    events_test  = y_test['event'].copy()

    # Cox-guided dysregulation selection on training only
    cox_pvals_dysreg = {}
    for gene in dysreg_train.columns:
        try:
            df_tmp = pd.DataFrame({'T': times_train, 'E': events_train,
                                   'gene': dysreg_train[gene].values})
            cph = CoxPHFitter()
            cph.fit(df_tmp, duration_col='T', event_col='E', show_progress=False)
            cox_pvals_dysreg[gene] = cph.summary['p'].values[0]
        except:
            cox_pvals_dysreg[gene] = 1.0
    top_dysreg_genes = pd.Series(cox_pvals_dysreg).nsmallest(20).index

    dysreg_train_sel = dysreg_train[top_dysreg_genes]
    dysreg_test_sel  = dysreg_test[top_dysreg_genes]

    scaler_expr     = StandardScaler()
    scaler_dysreg   = StandardScaler()
    scaler_immune   = StandardScaler()
    scaler_clinical = StandardScaler()

    expr_train_s     = scaler_expr.fit_transform(expr_train)
    expr_test_s      = scaler_expr.transform(expr_test)
    dysreg_train_s   = scaler_dysreg.fit_transform(dysreg_train_sel)
    dysreg_test_s    = scaler_dysreg.transform(dysreg_test_sel)
    immune_train_s   = scaler_immune.fit_transform(immune_train)
    immune_test_s    = scaler_immune.transform(immune_test)
    clinical_train_s = scaler_clinical.fit_transform(clinical_train)
    clinical_test_s  = scaler_clinical.transform(clinical_test)

    val_size     = int(0.2 * len(train_idx))
    val_expr     = torch.FloatTensor(expr_train_s[:val_size])
    val_dysreg   = torch.FloatTensor(dysreg_train_s[:val_size])
    val_immune   = torch.FloatTensor(immune_train_s[:val_size])
    val_clinical = torch.FloatTensor(clinical_train_s[:val_size])
    val_times    = times_train[:val_size].copy()
    val_events   = events_train[:val_size].copy()

    train_dataset = SurvivalDataset(
        expr_train_s, dysreg_train_s, immune_train_s, clinical_train_s,
        times_train.copy(), events_train.copy())
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Load model with lung pretrained encoder
    model = FusionModelPretrained(
        expr_dim=len(common_genes), dysreg_dim=20, immune_dim=22, clinical_dim=5
    ).to(device)
    model.load_pretrained_encoder(
        f'{base}/models/experiments/lung_pretrain/encoder_weights_lung.pt')

    model, best_val_ci, best_epoch = train_model(
        model, train_loader,
        val_expr, val_dysreg, val_immune, val_clinical,
        val_times, val_events,
        epochs=300, patience=30, lr=0.001, noise=0.05
    )

    model.eval()
    with torch.no_grad():
        test_risk, test_attn = model(
            torch.FloatTensor(expr_test_s).to(device),
            torch.FloatTensor(dysreg_test_s).to(device),
            torch.FloatTensor(immune_test_s).to(device),
            torch.FloatTensor(clinical_test_s).to(device)
        )

    test_risk = test_risk.squeeze().cpu().numpy()
    test_attn = test_attn.cpu().numpy()

    ci_test = concordance_index_censored(
        events_test.astype(bool), times_test, test_risk)[0]

    fold_cindex.append(ci_test)
    fold_attn_weights.append(test_attn)

    print(f"{fold:<6} {best_epoch:<12} {ci_test:.4f}")

print("-" * 32)
print(f"\nLung Pretrained Fusion C-index: {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")
print(f"\nFull comparison:")
print(f"  Cox Clinical:                    0.700")
print(f"  Cox-Lasso Expression:            0.649")
print(f"  Fusion V3 stable:                0.655 ± 0.032")
print(f"  Fusion V3 augmented:             0.669 ± 0.074")
print(f"  Fusion pan-cancer pretrain:      0.642 ± 0.076")
print(f"  Fusion lung pretrain:            {np.mean(fold_cindex):.3f} ± {np.std(fold_cindex):.3f}")

Patients:         478
Expression genes: 790
Events:           121 (25.3%)

Running leakage-free 5-fold CV — Lung Pretrained Fusion Model
Fold   Best Epoch   Test C-index
--------------------------------
Pretrained encoder weights loaded ✅
1      94           0.5994
Pretrained encoder weights loaded ✅
2      82           0.7371
Pretrained encoder weights loaded ✅
3      60           0.6515
Pretrained encoder weights loaded ✅
4      67           0.6138
Pretrained encoder weights loaded ✅
5      70           0.6955
--------------------------------

Lung Pretrained Fusion C-index: 0.659 ± 0.051

Full comparison:
  Cox Clinical:                    0.700
  Cox-Lasso Expression:            0.649
  Fusion V3 stable:                0.655 ± 0.032
  Fusion V3 augmented:             0.669 ± 0.074
  Fusion pan-cancer pretrain:      0.642 ± 0.076
  Fusion lung pretrain:            0.659 ± 0.051


In [12]:
os.makedirs(f'{base}/models/experiments/lung_pretrain_fusion', exist_ok=True)

torch.save(model.state_dict(), 
           f'{base}/models/experiments/lung_pretrain_fusion/fusion_lung_pretrain.pt')

results = {
    "model": "FusionModelPretrained",
    "pretrain": "lung-specific (LUAD + LUSC, 582 patients)",
    "cv_strategy": "StratifiedKFold_5fold",
    "augmentation": "Gaussian noise=0.05",
    "expr_genes": 790,
    "dysreg_genes": 20,
    "cv_cindex_mean": round(float(np.mean(fold_cindex)), 3),
    "cv_cindex_std": round(float(np.std(fold_cindex)), 3),
    "fold_cindices": [round(float(c), 4) for c in fold_cindex]
}

with open(f'{base}/models/experiments/lung_pretrain_fusion/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Saved: models/experiments/lung_pretrain_fusion/")
print(f"C-index: {results['cv_cindex_mean']} ± {results['cv_cindex_std']}")

Saved: models/experiments/lung_pretrain_fusion/
C-index: 0.659 ± 0.051
